1. Clones PlantDoc dataset
2. Checks dataset structure
3. Removes corrupted images
4. Creates train, validation, and test records
5. Applies data augmentation
6. Trains 5 models
7. Fine-tunes transfer learning models
8. Evaluates accuracy, precision, recall, F1-score
9. Calculates model size
10. Calculates training time
11. Calculates inference time per image
12. Generates confusion matrix
13. Generates comparison charts
14. Creates paper-ready result tables
15. Creates ZIP file of all results

PlantDoc introduced a real-world plant disease dataset,
but detailed comparison of modern transfer learning models
using both accuracy and practical deployment metrics
was limited.

Therefore, this study compares Custom CNN, MobileNetV2,
ResNet50, EfficientNetB0, and DenseNet121 on PlantDoc
using accuracy, precision, recall, F1-score, model size,
training time, and inference time.

# Comparative Deep Learning Analysis for Real-World Plant Disease Detection Using PlantDoc Dataset

This notebook compares five deep learning models on the PlantDoc dataset:

1. Custom CNN
2. MobileNetV2
3. ResNet50
4. EfficientNetB0
5. DenseNet121

Evaluation metrics:

- Accuracy
- Precision
- Recall
- F1-score
- Model size
- Training time
- Inference time per image

The objective is to identify both the best-performing model and the most practical model for low-resource or mobile deployment.

In [3]:
# ============================================================
# STEP 1: INSTALL AND IMPORT REQUIRED LIBRARIES
# ============================================================

!pip install tensorflow

import os
import time
import shutil
import random
import zipfile
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices("GPU"))

TensorFlow Version: 2.21.0
GPU Available: []


In [4]:
# ============================================================
# STEP 2: BASIC CONFIGURATION
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (224, 224)
BATCH_SIZE = 16

# For quick test, use fewer epochs.
# For final paper result, increase epochs.
CUSTOM_CNN_EPOCHS = 25
TRANSFER_INITIAL_EPOCHS = 10
TRANSFER_FINE_TUNE_EPOCHS = 10

RESULT_DIR = Path("/content/plantdoc_results")
MODEL_DIR = RESULT_DIR / "saved_models"
FIGURE_DIR = RESULT_DIR / "figures"

RESULT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Result directory:", RESULT_DIR)

Image size: (224, 224)
Batch size: 16
Result directory: /content/plantdoc_results


In [5]:
# ============================================================
# STEP 3: DOWNLOAD PLANTDOC DATASET
# ============================================================

# This is the public PlantDoc dataset repository.
# If this clone fails, manually upload dataset ZIP and extract it.

DATASET_REPO = "https://github.com/pratikkayal/PlantDoc-Dataset.git"
DATASET_DIR = Path("/content/PlantDoc-Dataset")

if not DATASET_DIR.exists():
    !git clone https://github.com/pratikkayal/PlantDoc-Dataset.git /content/PlantDoc-Dataset
else:
    print("Dataset already exists.")

print("Dataset downloaded/existing at:", DATASET_DIR)

Cloning into '/content/PlantDoc-Dataset'...
remote: Enumerating objects: 2670, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 2670 (delta 22), reused 22 (delta 22), pack-reused 2635 (from 1)
Receiving objects: 100% (2670/2670), 932.92 MiB | 43.99 MiB/s, done.
Resolving deltas: 100% (24/24), done.
Updating files: 100% (2581/2581), done.
Dataset downloaded/existing at: /content/PlantDoc-Dataset


In [6]:
# ============================================================
# STEP 4: FIND IMAGE FILES AND CLASS LABELS
# ============================================================

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def find_image_files(dataset_root):
    dataset_root = Path(dataset_root)
    records = []

    for img_path in dataset_root.rglob("*"):
        if img_path.suffix.lower() in image_extensions:
            label = img_path.parent.name.strip()
            records.append({
                "filepath": str(img_path),
                "label": label
            })

    return pd.DataFrame(records)

df = find_image_files(DATASET_DIR)

print("Total image files found:", len(df))
print("Sample records:")
display(df.head())

print("\nNumber of raw classes:", df["label"].nunique())
print("\nClass distribution:")
display(df["label"].value_counts().head(40))

Total image files found: 2579
Sample records:


,filepath,label
0,/content/PlantDoc-Dataset/PlantDoc_Examples.png,PlantDoc-Dataset
1,/content/PlantDoc-Dataset/train/Corn Gray leaf...,Corn Gray leaf spot
2,/content/PlantDoc-Dataset/train/Corn Gray leaf...,Corn Gray leaf spot
3,/content/PlantDoc-Dataset/train/Corn Gray leaf...,Corn Gray leaf spot
4,/content/PlantDoc-Dataset/train/Corn Gray leaf...,Corn Gray leaf spot



Number of raw classes: 29

Class distribution:


,count
label,
Corn leaf blight,192
Tomato Septoria leaf spot,151
Squash Powdery mildew leaf,130
Raspberry leaf,119
Blueberry leaf,117
Potato leaf early blight,117
Corn rust leaf,116
Peach leaf,112
Tomato leaf late blight,111


In [7]:
# ============================================================
# STEP 5: REMOVE CORRUPTED OR UNREADABLE IMAGES
# ============================================================

def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False

valid_flags = []

for path in df["filepath"]:
    valid_flags.append(is_valid_image(path))

df["valid"] = valid_flags

invalid_count = (~df["valid"]).sum()
print("Invalid images found:", invalid_count)

df = df[df["valid"]].drop(columns=["valid"]).reset_index(drop=True)

print("Images after cleaning:", len(df))
print("Classes after cleaning:", df["label"].nunique())

Invalid images found: 0
Images after cleaning: 2579
Classes after cleaning: 29


In [8]:



# ============================================================
# STEP 6: FILTER CLASSES WITH VERY FEW IMAGES
# ============================================================

# Keep classes with at least 10 images.
# You can increase this to 20 or 50 for stricter training.
MIN_IMAGES_PER_CLASS = 10

class_counts = df["label"].value_counts()
valid_classes = class_counts[class_counts >= MIN_IMAGES_PER_CLASS].index.tolist()

df = df[df["label"].isin(valid_classes)].reset_index(drop=True)

print("Images after class filtering:", len(df))
print("Final number of classes:", df["label"].nunique())

display(df["label"].value_counts())



Images after class filtering: 2576
Final number of classes: 27


,count
label,
Corn leaf blight,192
Tomato Septoria leaf spot,151
Squash Powdery mildew leaf,130
Raspberry leaf,119
Blueberry leaf,117
Potato leaf early blight,117
Corn rust leaf,116
Peach leaf,112
Tomato leaf late blight,111


In [9]:



# ============================================================
# STEP 7: ENCODE TEXT LABELS INTO NUMERIC LABELS
# ============================================================

label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])

class_names = list(label_encoder.classes_)
num_classes = len(class_names)

print("Number of classes:", num_classes)
print("Class names:")
for i, name in enumerate(class_names):
    print(i, ":", name)



Number of classes: 27
Class names:
0 : Apple Scab Leaf
1 : Apple leaf
2 : Apple rust leaf
3 : Bell_pepper leaf
4 : Bell_pepper leaf spot
5 : Blueberry leaf
6 : Cherry leaf
7 : Corn Gray leaf spot
8 : Corn leaf blight
9 : Corn rust leaf
10 : Peach leaf
11 : Potato leaf early blight
12 : Potato leaf late blight
13 : Raspberry leaf
14 : Soyabean leaf
15 : Squash Powdery mildew leaf
16 : Strawberry leaf
17 : Tomato Early blight leaf
18 : Tomato Septoria leaf spot
19 : Tomato leaf
20 : Tomato leaf bacterial spot
21 : Tomato leaf late blight
22 : Tomato leaf mosaic virus
23 : Tomato leaf yellow virus
24 : Tomato mold leaf
25 : grape leaf
26 : grape leaf black rot


In [10]:
# ============================================================
# STEP 8: CREATE TRAIN, VALIDATION, AND TEST SPLITS
# ============================================================

# Split:
# 70% training
# 15% validation
# 15% testing

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["label_id"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label_id"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Testing images:", len(test_df))

print("\nTrain class distribution:")
display(train_df["label"].value_counts().head())

Training images: 1803
Validation images: 386
Testing images: 387

Train class distribution:


,count
label,
Corn leaf blight,134
Tomato Septoria leaf spot,106
Squash Powdery mildew leaf,91
Raspberry leaf,83
Blueberry leaf,82


In [11]:
# ============================================================
# STEP 9: COMPUTE CLASS WEIGHTS
# ============================================================

classes = np.unique(train_df["label_id"])

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label_id"]
)

class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(classes, class_weights_array)
}

print("Class weights calculated.")

Class weights calculated.


In [12]:



# ============================================================
# STEP 10: DATA AUGMENTATION AND PREPROCESSING
# ============================================================

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name="data_augmentation")


def get_preprocess_function(model_name):
    if model_name == "CustomCNN":
        return lambda x: x / 255.0

    elif model_name == "MobileNetV2":
        return tf.keras.applications.mobilenet_v2.preprocess_input

    elif model_name == "ResNet50":
        return tf.keras.applications.resnet50.preprocess_input

    elif model_name == "EfficientNetB0":
        return tf.keras.applications.efficientnet.preprocess_input

    elif model_name == "DenseNet121":
        return tf.keras.applications.densenet.preprocess_input

    else:
        raise ValueError("Unknown model name")


def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    return image, label


def make_dataset(dataframe, model_name, training=False):
    paths = dataframe["filepath"].values
    labels = dataframe["label_id"].values.astype(np.int32)

    preprocess_fn = get_preprocess_function(model_name)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED)

    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)

    def augment_and_preprocess(image, label):
        if training:
            image = data_augmentation(image, training=True)

        image = preprocess_fn(image)
        return image, label

    ds = ds.map(augment_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds





In [13]:





# ============================================================
# STEP 11: CUSTOM CNN MODEL
# ============================================================

def build_custom_cnn(num_classes):
    model = models.Sequential([
        layers.Input(shape=(224, 224, 3)),

        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(256, (3, 3), padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),

        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),

        layers.Dense(num_classes, activation="softmax")
    ], name="CustomCNN")

    return model







In [15]:



# ============================================================
# STEP 12: TRANSFER LEARNING MODEL BUILDERS
# ============================================================

def build_transfer_model(model_name, num_classes):
    input_shape = (224, 224, 3)

    if model_name == "MobileNetV2":
        base_model = tf.keras.applications.MobileNetV2(
            include_top=False,
            weights="imagenet",
            input_shape=input_shape
        )

    elif model_name == "ResNet50":
        base_model = tf.keras.applications.ResNet50(
            include_top=False,
            weights="imagenet",
            input_shape=input_shape
        )

    elif model_name == "EfficientNetB0":
        base_model = tf.keras.applications.EfficientNetB0(
            include_top=False,
            weights="imagenet",
            input_shape=input_shape
        )

    elif model_name == "DenseNet121":
        base_model = tf.keras.applications.DenseNet121(
            include_top=False,
            weights="imagenet",
            input_shape=input_shape
        )

    else:
        raise ValueError("Unknown transfer model name")

    base_model.trainable = False

    inputs = layers.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs, name=model_name)

    return model, base_model




In [16]:




# ============================================================
# STEP 13: TRAINING AND EVALUATION FUNCTIONS
# ============================================================

def compile_model(model, learning_rate):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )


def get_callbacks(model_name):
    checkpoint_path = MODEL_DIR / f"{model_name}_best.keras"

    callbacks = [
        EarlyStopping(
            monitor="val_accuracy",
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.3,
            patience=3,
            min_lr=1e-7,
            verbose=1
        ),
        ModelCheckpoint(
            filepath=str(checkpoint_path),
            monitor="val_accuracy",
            save_best_only=True,
            verbose=1
        )
    ]

    return callbacks


def fine_tune_base_model(base_model, fine_tune_at=None):
    base_model.trainable = True

    if fine_tune_at is None:
        fine_tune_at = int(len(base_model.layers) * 0.70)

    for layer in base_model.layers[:fine_tune_at]:
        layer.trainable = False

    # Keep BatchNormalization layers frozen for stable fine-tuning
    for layer in base_model.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False


def evaluate_model(model, test_ds, model_name):
    y_true = test_df["label_id"].values

    start_time = time.time()
    y_prob = model.predict(test_ds, verbose=1)
    total_inference_time = time.time() - start_time

    y_pred = np.argmax(y_prob, axis=1)

    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    accuracy = accuracy_score(y_true, y_pred)
    precision_macro = report["macro avg"]["precision"]
    recall_macro = report["macro avg"]["recall"]
    f1_macro = report["macro avg"]["f1-score"]

    inference_time_per_image = total_inference_time / len(test_df)

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision_Macro": precision_macro,
        "Recall_Macro": recall_macro,
        "F1_Macro": f1_macro,
        "Total_Inference_Time_sec": total_inference_time,
        "Inference_Time_per_Image_sec": inference_time_per_image,
        "y_true": y_true,
        "y_pred": y_pred,
        "classification_report": report
    }





In [17]:



# ============================================================
# STEP 14: CONFUSION MATRIX FUNCTION
# ============================================================

def plot_confusion_matrix(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(18, 16))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"Confusion Matrix - {model_name}", fontsize=16)
    plt.colorbar()

    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=90, fontsize=8)
    plt.yticks(tick_marks, class_names, fontsize=8)

    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()

    save_path = FIGURE_DIR / f"{model_name}_confusion_matrix.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved:", save_path)




In [18]:



# ============================================================
# STEP 15: COMPLETE TRAINING PIPELINE FOR ONE MODEL
# ============================================================

def train_and_evaluate_model(model_name):
    print("\n" + "=" * 80)
    print("Training Model:", model_name)
    print("=" * 80)

    train_ds = make_dataset(train_df, model_name, training=True)
    val_ds = make_dataset(val_df, model_name, training=False)
    test_ds = make_dataset(test_df, model_name, training=False)

    if model_name == "CustomCNN":
        model = build_custom_cnn(num_classes)
        base_model = None

        compile_model(model, learning_rate=1e-3)

        start_train_time = time.time()

        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=CUSTOM_CNN_EPOCHS,
            class_weight=class_weights,
            callbacks=get_callbacks(model_name),
            verbose=1
        )

        training_time = time.time() - start_train_time

    else:
        model, base_model = build_transfer_model(model_name, num_classes)

        compile_model(model, learning_rate=1e-3)

        start_train_time = time.time()

        print("\nInitial training with frozen base model...")

        history_initial = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=TRANSFER_INITIAL_EPOCHS,
            class_weight=class_weights,
            callbacks=get_callbacks(model_name + "_initial"),
            verbose=1
        )

        print("\nFine-tuning selected layers...")

        fine_tune_base_model(base_model)

        compile_model(model, learning_rate=1e-5)

        history_fine = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=TRANSFER_FINE_TUNE_EPOCHS,
            class_weight=class_weights,
            callbacks=get_callbacks(model_name + "_finetune"),
            verbose=1
        )

        training_time = time.time() - start_train_time

        # combine history manually
        history = {
            "initial": history_initial.history,
            "fine_tune": history_fine.history
        }

    # Save final model
    final_model_path = MODEL_DIR / f"{model_name}_final.keras"
    model.save(final_model_path)

    model_size_mb = os.path.getsize(final_model_path) / (1024 * 1024)
    total_params = model.count_params()

    eval_result = evaluate_model(model, test_ds, model_name)

    eval_result["Training_Time_sec"] = training_time
    eval_result["Training_Time_min"] = training_time / 60
    eval_result["Model_Size_MB"] = model_size_mb
    eval_result["Total_Parameters"] = total_params
    eval_result["Saved_Model_Path"] = str(final_model_path)

    print("\nEvaluation Summary:")
    print("Accuracy:", eval_result["Accuracy"])
    print("Precision Macro:", eval_result["Precision_Macro"])
    print("Recall Macro:", eval_result["Recall_Macro"])
    print("F1 Macro:", eval_result["F1_Macro"])
    print("Model Size MB:", eval_result["Model_Size_MB"])
    print("Training Time min:", eval_result["Training_Time_min"])
    print("Inference Time/Image sec:", eval_result["Inference_Time_per_Image_sec"])

    plot_confusion_matrix(
        eval_result["y_true"],
        eval_result["y_pred"],
        model_name
    )

    del model
    gc.collect()
    tf.keras.backend.clear_session()

    return eval_result





In [ ]:




# ============================================================
# STEP 16: TRAIN AND EVALUATE ALL FIVE MODELS
# ============================================================

models_to_train = [
    "CustomCNN",
    "MobileNetV2",
    "ResNet50",
    "EfficientNetB0",
    "DenseNet121"
]

all_results = []

for model_name in models_to_train:
    result = train_and_evaluate_model(model_name)
    all_results.append(result)

print("All models trained and evaluated.")






Training Model: CustomCNN
Epoch 1/25
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.0606 - loss: 3.6299
Epoch 1: val_accuracy improved from None to 0.04404, saving model to /content/plantdoc_results/saved_models/CustomCNN_best.keras

Epoch 1: finished saving model to /content/plantdoc_results/saved_models/CustomCNN_best.keras
113/113 ━━━━━━━━━━━━━━━━━━━━ 159s 1s/step - accuracy: 0.0638 - loss: 3.4892 - val_accuracy: 0.0440 - val_loss: 3.8938 - learning_rate: 0.0010
Epoch 2/25
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1080 - loss: 3.2414
Epoch 2: val_accuracy did not improve from 0.04404
113/113 ━━━━━━━━━━━━━━━━━━━━ 163s 1s/step - accuracy: 0.0943 - loss: 3.2332 - val_accuracy: 0.0440 - val_loss: 4.1067 - learning_rate: 0.0010
Epoch 3/25
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1029 - loss: 3.1409
Epoch 3: val_accuracy improved from 0.04404 to 0.08031, saving model to /content/plantdoc_results/saved_models/CustomCNN_best.keras

Epoch 3: finished saving m

In [ ]:


# ============================================================
# STEP 17: CREATE FINAL RESULT TABLE
# ============================================================

summary_rows = []

for result in all_results:
    summary_rows.append({
        "Model": result["Model"],
        "Accuracy": result["Accuracy"],
        "Precision_Macro": result["Precision_Macro"],
        "Recall_Macro": result["Recall_Macro"],
        "F1_Macro": result["F1_Macro"],
        "Model_Size_MB": result["Model_Size_MB"],
        "Total_Parameters": result["Total_Parameters"],
        "Training_Time_min": result["Training_Time_min"],
        "Inference_Time_per_Image_sec": result["Inference_Time_per_Image_sec"]
    })

results_df = pd.DataFrame(summary_rows)

# Sort by F1-score first
results_df = results_df.sort_values(by="F1_Macro", ascending=False).reset_index(drop=True)

display(results_df)

csv_path = RESULT_DIR / "five_model_comparative_analysis_results.csv"
results_df.to_csv(csv_path, index=False)

print("Saved CSV:", csv_path)




In [ ]:


# ============================================================
# STEP 18: CREATE RESULT TABLE
# ============================================================

paper_table = results_df.copy()

paper_table["Accuracy (%)"] = paper_table["Accuracy"] * 100
paper_table["Precision (%)"] = paper_table["Precision_Macro"] * 100
paper_table["Recall (%)"] = paper_table["Recall_Macro"] * 100
paper_table["F1-score (%)"] = paper_table["F1_Macro"] * 100

paper_table = paper_table[[
    "Model",
    "Accuracy (%)",
    "Precision (%)",
    "Recall (%)",
    "F1-score (%)",
    "Model_Size_MB",
    "Training_Time_min",
    "Inference_Time_per_Image_sec",
    "Total_Parameters"
]]

paper_table = paper_table.round(4)

display(paper_table)

paper_csv_path = RESULT_DIR / "ieee_paper_result_table.csv"
paper_table.to_csv(paper_csv_path, index=False)

print("Saved IEEE paper table:", paper_csv_path)




In [ ]:



# ============================================================
# STEP 19: ACCURACY COMPARISON GRAPH
# ============================================================

plt.figure(figsize=(10, 6))
plt.bar(results_df["Model"], results_df["Accuracy"] * 100)
plt.xlabel("Model")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Comparison of Deep Learning Models")
plt.xticks(rotation=30)
plt.tight_layout()

save_path = FIGURE_DIR / "accuracy_comparison.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", save_path)





In [ ]:




# ============================================================
# STEP 20: F1-SCORE COMPARISON GRAPH
# ============================================================

plt.figure(figsize=(10, 6))
plt.bar(results_df["Model"], results_df["F1_Macro"] * 100)
plt.xlabel("Model")
plt.ylabel("Macro F1-score (%)")
plt.title("Macro F1-score Comparison of Deep Learning Models")
plt.xticks(rotation=30)
plt.tight_layout()

save_path = FIGURE_DIR / "f1_score_comparison.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", save_path)





In [ ]:




# ============================================================
# STEP 21: MODEL SIZE COMPARISON GRAPH
# ============================================================

plt.figure(figsize=(10, 6))
plt.bar(results_df["Model"], results_df["Model_Size_MB"])
plt.xlabel("Model")
plt.ylabel("Model Size (MB)")
plt.title("Model Size Comparison")
plt.xticks(rotation=30)
plt.tight_layout()

save_path = FIGURE_DIR / "model_size_comparison.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", save_path)






In [ ]:



# ============================================================
# STEP 22: TRAINING TIME COMPARISON GRAPH
# ============================================================

plt.figure(figsize=(10, 6))
plt.bar(results_df["Model"], results_df["Training_Time_min"])
plt.xlabel("Model")
plt.ylabel("Training Time (minutes)")
plt.title("Training Time Comparison")
plt.xticks(rotation=30)
plt.tight_layout()

save_path = FIGURE_DIR / "training_time_comparison.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", save_path)






In [ ]:



# ============================================================
# STEP 23: INFERENCE TIME COMPARISON GRAPH
# ============================================================

plt.figure(figsize=(10, 6))
plt.bar(results_df["Model"], results_df["Inference_Time_per_Image_sec"])
plt.xlabel("Model")
plt.ylabel("Inference Time per Image (seconds)")
plt.title("Inference Time Comparison")
plt.xticks(rotation=30)
plt.tight_layout()

save_path = FIGURE_DIR / "inference_time_comparison.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", save_path)






In [ ]:




# ============================================================
# STEP 24: ACCURACY VS MODEL SIZE TRADE-OFF GRAPH
# ============================================================

plt.figure(figsize=(10, 6))

plt.scatter(
    results_df["Model_Size_MB"],
    results_df["Accuracy"] * 100,
    s=120
)

for i, row in results_df.iterrows():
    plt.text(
        row["Model_Size_MB"],
        row["Accuracy"] * 100,
        row["Model"],
        fontsize=9
    )

plt.xlabel("Model Size (MB)")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy vs Model Size Trade-off")
plt.tight_layout()

save_path = FIGURE_DIR / "accuracy_vs_model_size_tradeoff.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", save_path)





In [ ]:



# ============================================================
# STEP 25: F1-SCORE VS INFERENCE TIME TRADE-OFF GRAPH
# ============================================================

plt.figure(figsize=(10, 6))

plt.scatter(
    results_df["Inference_Time_per_Image_sec"],
    results_df["F1_Macro"] * 100,
    s=120
)

for i, row in results_df.iterrows():
    plt.text(
        row["Inference_Time_per_Image_sec"],
        row["F1_Macro"] * 100,
        row["Model"],
        fontsize=9
    )

plt.xlabel("Inference Time per Image (seconds)")
plt.ylabel("Macro F1-score (%)")
plt.title("F1-score vs Inference Time Trade-off")
plt.tight_layout()

save_path = FIGURE_DIR / "f1_vs_inference_time_tradeoff.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", save_path)





In [ ]:




# ============================================================
# STEP 26: SAVE CLASS-WISE CLASSIFICATION REPORTS
# ============================================================

for result in all_results:
    model_name = result["Model"]
    report_df = pd.DataFrame(result["classification_report"]).transpose()

    report_path = RESULT_DIR / f"{model_name}_classification_report.csv"
    report_df.to_csv(report_path)

    print("Saved:", report_path)





In [ ]:



# ============================================================
# STEP 27: IDENTIFY BEST MODELS
# ============================================================

best_accuracy_model = results_df.sort_values(by="Accuracy", ascending=False).iloc[0]
best_f1_model = results_df.sort_values(by="F1_Macro", ascending=False).iloc[0]
smallest_model = results_df.sort_values(by="Model_Size_MB", ascending=True).iloc[0]
fastest_model = results_df.sort_values(by="Inference_Time_per_Image_sec", ascending=True).iloc[0]

print("Best Accuracy Model:")
print(best_accuracy_model[["Model", "Accuracy"]])

print("\nBest F1-score Model:")
print(best_f1_model[["Model", "F1_Macro"]])

print("\nSmallest Model:")
print(smallest_model[["Model", "Model_Size_MB"]])

print("\nFastest Inference Model:")
print(fastest_model[["Model", "Inference_Time_per_Image_sec"]])





In [ ]:




# ============================================================
# STEP 28: GENERATE PAPER-READY RESULT INTERPRETATION
# ============================================================

best_acc_name = best_accuracy_model["Model"]
best_acc_value = best_accuracy_model["Accuracy"] * 100

best_f1_name = best_f1_model["Model"]
best_f1_value = best_f1_model["F1_Macro"] * 100

small_name = smallest_model["Model"]
small_size = smallest_model["Model_Size_MB"]

fast_name = fastest_model["Model"]
fast_time = fastest_model["Inference_Time_per_Image_sec"]

interpretation = f"""
The experimental results show that {best_acc_name} achieved the highest classification accuracy of {best_acc_value:.2f}% on the PlantDoc test set.
In terms of macro F1-score, {best_f1_name} performed best with an F1-score of {best_f1_value:.2f}%.
However, practical deployment requires not only high accuracy but also low model size and faster inference.
The smallest model was {small_name}, with a saved model size of {small_size:.2f} MB.
The fastest model in terms of inference time was {fast_name}, requiring only {fast_time:.5f} seconds per image.

These findings indicate that the best model for accuracy may not always be the best model for low-resource deployment.
Therefore, this comparative analysis helps identify both the most accurate model and the most practical model for real-world plant disease detection.
"""

print(interpretation)

text_path = RESULT_DIR / "paper_ready_interpretation.txt"
with open(text_path, "w") as f:
    f.write(interpretation)

print("Saved interpretation:", text_path)






In [ ]:



# ============================================================
# STEP 29: ZIP ALL RESULTS FOR DOWNLOAD
# ============================================================

zip_output = "/content/plantdoc_5_model_results.zip"

if os.path.exists(zip_output):
    os.remove(zip_output)

shutil.make_archive(
    base_name="/content/plantdoc_5_model_results",
    format="zip",
    root_dir=str(RESULT_DIR)
)

print("ZIP file created:", zip_output)





In [ ]:
# ============================================================
# STEP 30: DOWNLOAD RESULT ZIP
# ============================================================

from google.colab import files

files.download("/content/plantdoc_5_model_results.zip")

